# Day 1: data, care reward model, pre-trained baseline

Runtime > Change runtime type > **T4 GPU** before running.

Before starting, upload `MIC.csv` to `MyDrive/nlp-moral-lora/data/MIC.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

### Clone the repo
If the repo is private, paste a GitHub personal access token when prompted (it isn't printed or saved).

In [ ]:
import getpass, os
GITHUB_USER = 'natitedros'
REPO = 'CMVA-context-based-aggregation'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'
assert os.path.exists(os.environ['PROJECT_ROOT'] + '/data/MIC.csv'), 'Upload MIC.csv to data/ first'

### 1. Build the per-value splits
Check the printed column names and label counts. If it complains about missing columns, edit the `COL_*` names at the top of `src/prepare_data.py`.

In [ ]:
!python src/prepare_data.py

### 2. Train the care reward classifier (~5-10 min on a T4)
Compare `test_macro_f1` with `majority_class_accuracy`. Macro-F1 around 0.7+ is usable.

In [ ]:
!python src/train_reward.py --value care

### 3. Pre-trained Qwen answers 20 held-out questions, scored for care
This is your first "PT model" number. Scale `--n` up to 500 once everything works.

In [ ]:
!python src/generate_and_score.py --tag pt_smoke --n 20 --values care

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 200)
pd.read_csv(os.environ['PROJECT_ROOT'] + '/results/generations_pt_smoke.csv').head(10)

### 4. If all of that worked: the other four reward models
Each takes a few minutes. Results land in `results/reward_{value}.json`.

In [ ]:
for v in ['fairness', 'loyalty', 'authority', 'sanctity']:
    !python src/train_reward.py --value {v}